In [1]:
!pip install "flwr[simulation]" torch torchvision numpy

"""
Federated Learning on MNIST using Flower (flwr) — Current API (2025)
Works in: VS Code (Python 3.8+) and Google Colab

Install once:
    pip install "flwr[simulation]" torch torchvision numpy

Run:
    python federated_mnist.py
"""

# ─────────────────────────────────────────────
# 0. IMPORTS
# ─────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from collections import OrderedDict
import numpy as np

from flwr.client import NumPyClient, ClientApp
from flwr.common import ndarrays_to_parameters, Context
from flwr.server import ServerApp, ServerConfig, ServerAppComponents
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation

# ─────────────────────────────────────────────
# 1. MODEL  (simple CNN for MNIST)
# ─────────────────────────────────────────────
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3)   # 28x28 → 26x26
        self.conv2 = nn.Conv2d(32, 64, 3)  # 13x13 → 11x11
        self.fc1   = nn.Linear(64 * 5 * 5, 128)
        self.fc2   = nn.Linear(128, 10)
        self.pool  = nn.MaxPool2d(2, 2)
        self.relu  = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))  # → 13x13
        x = self.pool(self.relu(self.conv2(x)))  # → 5x5
        x = x.view(-1, 64 * 5 * 5)
        x = self.relu(self.fc1(x))
        return self.fc2(x)


# ─────────────────────────────────────────────
# 2. HELPERS  (weights ↔ numpy)
# ─────────────────────────────────────────────
def get_weights(model):
    return [val.cpu().numpy() for val in model.state_dict().values()]

def set_weights(model, weights):
    state = OrderedDict(
        {k: torch.tensor(v) for k, v in zip(model.state_dict().keys(), weights)}
    )
    model.load_state_dict(state, strict=True)


# ─────────────────────────────────────────────
# 3. DATASET  (download once, split across clients)
# ─────────────────────────────────────────────
NUM_CLIENTS = 10
BATCH_SIZE  = 32

transform = transforms.Compose([transforms.ToTensor()])

trainset = datasets.MNIST(".", train=True,  transform=transform, download=True)
testset  = datasets.MNIST(".", train=False, transform=transform, download=True)

# IID split: each client gets an equal random slice of the training data
indices = torch.randperm(len(trainset)).tolist()
partition_size = len(trainset) // NUM_CLIENTS
partitions = [
    indices[i * partition_size : (i + 1) * partition_size]
    for i in range(NUM_CLIENTS)
]


def get_loaders(client_id: int):
    """Return (trainloader, testloader) for a given client id."""
    train_subset = Subset(trainset, partitions[client_id])
    trainloader  = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    testloader   = DataLoader(testset,      batch_size=BATCH_SIZE)
    return trainloader, testloader


# ─────────────────────────────────────────────
# 4. FLOWER CLIENT
# ─────────────────────────────────────────────
class FlowerClient(NumPyClient):
    def __init__(self, trainloader, testloader):
        self.model       = Net()
        self.trainloader = trainloader
        self.testloader  = testloader
        self.criterion   = nn.CrossEntropyLoss()
        self.optimizer   = torch.optim.Adam(self.model.parameters(), lr=0.001)

    def get_parameters(self, config):
        return get_weights(self.model)

    def fit(self, parameters, config):
        set_weights(self.model, parameters)          # 1. load global model
        self.model.train()
        for images, labels in self.trainloader:      # 2. local training (1 epoch)
            self.optimizer.zero_grad()
            loss = self.criterion(self.model(images), labels)
            loss.backward()
            self.optimizer.step()
        return get_weights(self.model), len(self.trainloader.dataset), {}  # 3. return updated weights

    def evaluate(self, parameters, config):
        set_weights(self.model, parameters)
        self.model.eval()
        correct, total, loss_sum = 0, 0, 0.0
        with torch.no_grad():
            for images, labels in self.testloader:
                outputs = self.model(images)
                loss_sum += self.criterion(outputs, labels).item()
                predicted = outputs.argmax(dim=1)
                correct  += (predicted == labels).sum().item()
                total    += labels.size(0)
        accuracy = correct / total
        avg_loss = loss_sum / len(self.testloader)
        return avg_loss, total, {"accuracy": accuracy}


# ─────────────────────────────────────────────
# 5. ClientApp  (current Flower API)
# ─────────────────────────────────────────────
def client_fn(context: Context):
    """Called by the simulation engine for each virtual client."""
    client_id   = int(context.node_config["partition-id"])
    trainloader, testloader = get_loaders(client_id)
    return FlowerClient(trainloader, testloader).to_client()

client = ClientApp(client_fn=client_fn)


# ─────────────────────────────────────────────
# 6. ServerApp  (current Flower API)
# ─────────────────────────────────────────────
NUM_ROUNDS = 5

# Give the server an initial model so FedAvg can evaluate from round 0
initial_params = ndarrays_to_parameters(get_weights(Net()))

strategy = FedAvg(
    fraction_fit          = 0.4,   # 40 % of clients train each round
    fraction_evaluate     = 0.2,   # 20 % evaluate
    min_fit_clients       = 4,
    min_evaluate_clients  = 2,
    min_available_clients = NUM_CLIENTS,
    initial_parameters    = initial_params,
)

def server_fn(context: Context):
    return ServerAppComponents(
        strategy = strategy,
        config   = ServerConfig(num_rounds=NUM_ROUNDS),
    )

server = ServerApp(server_fn=server_fn)


# ─────────────────────────────────────────────
# 7. RUN SIMULATION
# ─────────────────────────────────────────────
if __name__ == "__main__":
    run_simulation(
        server_app  = server,
        client_app  = client,
        num_supernodes = NUM_CLIENTS,
        backend_config = {"client_resources": {"num_cpus": 1, "num_gpus": 0.0}},
    )
    # Expected: ~96–98 % accuracy after 5 rounds

  0%|          | 0.00/9.91M [00:00<?, ?B/s]/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
100%|██████████| 9.91M/9.91M [00:00<00:00, 105MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 22.0MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 33.7MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.7MB/s]

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
DEBUG:flwr:backend_config: {'client_resources': {'num_cpus': 1, 'num_gpus': 0.0}}
DEBUG:flwr:Asyncio event loop already running.
/usr/local/lib/python3.12/dist-packages/jupyter_clie